# Simulating Future Waves in Sri Lanka
see the change in wave 

In [ ]:
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
# Configure
FUTUREWAVE_DIR = r'c:\Users\pba003\Documents\00projects\future_waves'

# Script to plot per-model wave metrics (individual points) computed by ensemble_comp.py
SCENARIOS = ['historical', 'ssp126', 'ssp585']

# categorical color per model, fixed order (validated palette slots 1-8)
CATEGORICAL_HUES = ['#2a78d6', '#008300', '#e87ba4', '#eda100', '#1baf7a', '#eb6834', '#4a3aa7', '#e34948']


In [ ]:
# load the most recently computed run
run_dirs = sorted(glob.glob(os.path.join(FUTUREWAVE_DIR, 'output/*_ensamble_*')), key=os.path.getmtime)
out_dir = run_dirs[-1]
df_out = pd.read_csv(os.path.join(out_dir, 'df_out.csv'))

# set up variable to plot
era5_row = df_out.loc[df_out['Model'] == 'ERA5'].iloc[0]
df_models = df_out.loc[df_out['Model'] != 'ERA5']
models = sorted(df_models['Model'].unique())  # fixed order so a model always gets the same color
if len(models) > len(CATEGORICAL_HUES):
    raise ValueError(f'{len(models)} models exceeds the {len(CATEGORICAL_HUES)} validated categorical hues; fold extras into facets instead of adding more colors.')
model_colors = dict(zip(models, CATEGORICAL_HUES))

METRIC_COLS = ['Hs50', 'Hs95', 'Nex', 'Dur50', 'Dir50', 'Dir90', 'Gap50', 'Energy']
METRIC_LABEL = ['Hs50 (m)', 'Hs95 (m)', 'N Stroms (Hs>Hs95_hist)', 'Median Duration (hr)', 'Dir50 (degree)', 'Dir90 (degree)', 'Gap50 (days)', 'Cumulative Energy (MJh/m2)']

In [ ]:
# plot scatter of each metric by model

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
# historical stays apart; ssp126/ssp585 are pulled closer together as a "future" cluster
scenario_x = np.array([0.0, 1.0, 1.5])
era5_x = scenario_x[SCENARIOS.index('historical')]
n_models = len(models)
jitter = np.linspace(-0.08, 0.08, n_models) if n_models > 1 else np.array([0.0])

for ax, metric, label in zip(axes.flat, METRIC_COLS, METRIC_LABEL):
    # shade the future scenarios to set them apart from the historical baseline
    future_start = (era5_x + scenario_x[SCENARIOS.index('ssp126')]) / 2
    ax.axvspan(future_start, scenario_x[-1] + 0.25, color='#f0efec', zorder=0)

    for i, model in enumerate(models):
        sub = df_models[df_models['Model'] == model].set_index('Scenario').reindex(SCENARIOS)
        ax.scatter(
            scenario_x + jitter[i], sub[metric].values, s=50,
            facecolor=model_colors[model], edgecolor='none', zorder=3, label=model,
        )

    ax.scatter(
        era5_x, era5_row[metric], marker='D', s=70,
        facecolor='#fcfcfb', edgecolor='#0b0b0b', linewidth=1.4, zorder=5,
    )

    ax.set_title(metric)
    ax.set_xticks(scenario_x)
    ax.set_xticklabels(SCENARIOS)
    ax.set_xlim(scenario_x[0] - 0.5, scenario_x[-1] + 0.5)
    ax.set_ylabel(label)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', markersize=8,
           markerfacecolor=model_colors[m], markeredgecolor='none', label=m)
    for m in models
]
legend_handles.append(Line2D(
    [0], [0], marker='D', linestyle='none', markersize=8,
    markerfacecolor='#fcfcfb', markeredgecolor='#0b0b0b', markeredgewidth=1.4,
    label='ERA5 (baseline)',
))
fig.suptitle('Ensemble wave metrics by model: present time slice (1985-2015) vs. future time slice (2071-2100)', y=0.995)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=len(legend_handles), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(os.path.join(out_dir, 'ensemble_metrics_scatter.png'), dpi=150)
plt.show()


In [ ]:
# plot the result: ensemble spread per scenario, box = range across models
METRIC_COLS = ['Hs50', 'Hs95', 'Nex', 'DurAvg', 'Dur50', 'Dir50', 'Gap50', 'Energy']
METRIC_LABEL = ['Hs50 (m)', 'Hs95 (m)', 'N Stroms (Hs>Hs95_hist)', 'Mean Duration (hr)', 'Median Duration (hr)', 'Dir50 (degree)', 'Gap50 (days)', 'Cumulative Energy (MJh/m2)']

# historical = neutral baseline; ssp126/ssp585 = low/high-emission futures (diverging blue/red)
SCENARIO_FACE = {'historical': '#c3c2b7', 'ssp126': '#2a78d6', 'ssp585': '#e34948'}
SCENARIO_EDGE = {'historical': '#898781', 'ssp126': '#184f95', 'ssp585': '#a53332'}

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
scenario_x = np.arange(len(SCENARIOS))
era5_x = scenario_x[SCENARIOS.index('historical')]

def minmax_stats(values):
    lo, hi = np.min(values), np.max(values)
    return {'med': np.median(values), 'q1': lo, 'q3': hi, 'whislo': lo, 'whishi': hi, 'fliers': []}

for ax, metric, label in zip(axes.flat, METRIC_COLS, METRIC_LABEL):
    data = [df_models.loc[df_models['Scenario'] == s, metric].values for s in SCENARIOS]
    bp = ax.bxp(
        [minmax_stats(d) for d in data], positions=scenario_x, widths=0.5, patch_artist=True,
        medianprops=dict(color='#0b0b0b', linewidth=1.5),
        whiskerprops=dict(color='#52514e'),
        capprops=dict(color='#52514e'),
        showfliers=False,
    )
    for patch, scenario in zip(bp['boxes'], SCENARIOS):
        patch.set_facecolor(SCENARIO_FACE[scenario])
        patch.set_edgecolor(SCENARIO_EDGE[scenario])
        patch.set_linewidth(1.2)

    ax.scatter(
        era5_x, era5_row[metric], marker='D', s=70,
        facecolor='#fcfcfb', edgecolor='#0b0b0b', linewidth=1.4, zorder=5, alpha=0.5
    )

    ax.set_title(metric)
    ax.set_xticks(scenario_x)
    ax.set_xticklabels(SCENARIOS)
    ax.set_ylabel(label)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

legend_handles = [Patch(facecolor=SCENARIO_FACE[s], edgecolor=SCENARIO_EDGE[s], label=s) for s in SCENARIOS]
legend_handles.append(Line2D(
    [0], [0], marker='D', linestyle='none', markersize=8,
    markerfacecolor='#fcfcfb', markeredgecolor='#0b0b0b', markeredgewidth=1.4,
    label='ERA5 (baseline)',
))
fig.suptitle('Ensemble wave metrics: present time slice (1985-2015) vs. future time slice (2071-2100)', y=0.995)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=len(legend_handles), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(os.path.join(out_dir, 'ensemble_metrics.png'), dpi=150)
plt.show()


In [ ]:
# get the columns to normalize
value_cols = df_out.columns.difference(['Scenario', 'Model'])

# get the historical value
hist = (
    df_out[df_out['Scenario'] == 'historical']
    .set_index('Model')[value_cols]
)

# create dataframe with historical values aligned to model 
hist_aligned = df_out['Model'].map(hist.to_dict(orient='index')).apply(pd.Series)
# hist_aligned.columns = value_cols
hist_aligned.index = df_out.index

# normalize the values
df_norm = df_out.copy()
df_norm2 = df_out.copy()

df_norm[value_cols] = df_out[value_cols] - hist_aligned[value_cols]
df_norm2[value_cols] = df_out[value_cols] / hist_aligned[value_cols]


In [ ]:
# set up variable to plot
era5_row = df_norm.loc[df_norm['Model'] == 'ERA5'].iloc[0]
df_models = df_norm.loc[df_norm['Model'] != 'ERA5']
models = sorted(df_models['Model'].unique())  # fixed order so a model always gets the same color
if len(models) > len(CATEGORICAL_HUES):
    raise ValueError(f'{len(models)} models exceeds the {len(CATEGORICAL_HUES)} validated categorical hues; fold extras into facets instead of adding more colors.')
model_colors = dict(zip(models, CATEGORICAL_HUES))

METRIC_COLS = ['Hs50', 'Hs95', 'Nex', 'Dur50', 'Dir50', 'Dir90', 'Gap50', 'Energy']
METRIC_LABEL = [r'$\Delta$ Hs50 (m)', r'$\Delta$ Hs95 (m)', r'$\Delta$ N Stroms', r'$\Delta$ Median Duration (hr)', r'$\Delta$ Dir50 (degree)', r'$\Delta$ Dir90 (degree)', r'$\Delta$ Gap50 (days)', r'$\Delta$ Cumulative Energy (MJh/m2)']

# plot scatter

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
# historical stays apart; ssp126/ssp585 are pulled closer together as a "future" cluster
scenario_x = np.array([0.0, 1.0, 1.5])
era5_x = scenario_x[SCENARIOS.index('historical')]
n_models = len(models)
jitter = np.linspace(-0.08, 0.08, n_models) if n_models > 1 else np.array([0.0])

for ax, metric, label in zip(axes.flat, METRIC_COLS, METRIC_LABEL):
    # shade the future scenarios to set them apart from the historical baseline
    max_offset = np.nanmax(np.abs(df_models[metric])) * 1.15
    ax.axhspan(0, max_offset, color='#D6E4F0', zorder=0)
    ax.axhspan(-max_offset, 0, color='#F5D6D6', zorder=0)

    for i, model in enumerate(models):
        sub = df_models[df_models['Model'] == model].set_index('Scenario').reindex(SCENARIOS)
        ax.scatter(
            scenario_x + jitter[i], sub[metric].values, s=50,
            facecolor=model_colors[model], edgecolor='none', zorder=3, label=model,
        )

        ax.plot(
            scenario_x + jitter[i], sub[metric].values, 
            color=model_colors[model], alpha=0.2, zorder=2, linestyle='--'
        )


    ax.set_title(metric)
    ax.set_xticks(scenario_x)
    ax.set_xticklabels(SCENARIOS)
    ax.set_xlim(scenario_x[0] - 0.5, scenario_x[-1] + 0.5)
    ax.set_ylim([-max_offset, max_offset])
    ax.set_ylabel(label)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', markersize=8,
           markerfacecolor=model_colors[m], markeredgecolor='none', label=m)
    for m in models
]
# legend_handles.append(Line2D(
#     [0], [0], marker='D', linestyle='none', markersize=8,
#     markerfacecolor='#fcfcfb', markeredgecolor='#0b0b0b', markeredgewidth=1.4,
#     label='ERA5 (baseline)',
# ))
# fig.suptitle('Ensemble wave metrics by model: present time slice (1985-2015) vs. future time slice (2071-2100)', y=0.995)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=len(legend_handles), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(os.path.join(out_dir, 'ensemble_metrics_scatter.png'), dpi=150)
plt.show()


In [ ]:
# set up variable to plot
df_models = df_norm2.loc[df_norm2['Model'] != 'ERA5']
models = sorted(df_models['Model'].unique())  # fixed order so a model always gets the same color
if len(models) > len(CATEGORICAL_HUES):
    raise ValueError(f'{len(models)} models exceeds the {len(CATEGORICAL_HUES)} validated categorical hues; fold extras into facets instead of adding more colors.')
model_colors = dict(zip(models, CATEGORICAL_HUES))

METRIC_COLS = ['Hs50', 'Hs95', 'Nex', 'Dur50', 'Dir50', 'Dir90', 'Gap50', 'Energy']
METRIC_LABEL = ['Ratio Hs50 [-]', 'Ratio Hs95 [-]', 'Ratio N Stroms [-]', 'Ratio Median Duration [-]', 'Ratio Dir50 [-]', 'Ratio Dir90 [-]', 'Ratio Gap50 [-]', 'Ratio Cumulative Energy [-]']

# plot scatter

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
# historical stays apart; ssp126/ssp585 are pulled closer together as a "future" cluster
scenario_x = np.array([0.0, 1.0, 1.5])
era5_x = scenario_x[SCENARIOS.index('historical')]
n_models = len(models)
jitter = np.linspace(-0.08, 0.08, n_models) if n_models > 1 else np.array([0.0])

for ax, metric, label in zip(axes.flat, METRIC_COLS, METRIC_LABEL):
    # shade the future scenarios to set them apart from the historical baseline
    max_offset = np.nanmax(np.abs(df_models[metric]-1)) * 1.15
    ax.axhspan(1, 1+max_offset, color='#D6E4F0', zorder=0)
    ax.axhspan((1-max_offset), 1, color='#F5D6D6', zorder=0)

    for i, model in enumerate(models):
        sub = df_models[df_models['Model'] == model].set_index('Scenario').reindex(SCENARIOS)
        ax.scatter(
            scenario_x + jitter[i], sub[metric].values, s=50,
            facecolor=model_colors[model], edgecolor='none', zorder=3, label=model,
        )

        ax.plot(
            scenario_x + jitter[i], sub[metric].values, 
            color=model_colors[model], alpha=0.2, zorder=2, linestyle='--'
        )


    ax.set_title(metric)
    ax.set_xticks(scenario_x)
    ax.set_xticklabels(SCENARIOS)
    ax.set_xlim(scenario_x[0] - 0.5, scenario_x[-1] + 0.5)
    ax.set_ylim([1-max_offset, 1+max_offset])
    ax.set_ylabel(label)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', markersize=8,
           markerfacecolor=model_colors[m], markeredgecolor='none', label=m)
    for m in models
]
# legend_handles.append(Line2D(
#     [0], [0], marker='D', linestyle='none', markersize=8,
#     markerfacecolor='#fcfcfb', markeredgecolor='#0b0b0b', markeredgewidth=1.4,
#     label='ERA5 (baseline)',
# ))
# fig.suptitle('Ensemble wave metrics by model: present time slice (1985-2015) vs. future time slice (2071-2100)', y=0.995)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=len(legend_handles), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(os.path.join(out_dir, 'ensemble_metrics_scatter.png'), dpi=150)
plt.show()
